# Phase 7 — Hiệu Chuẩn & Đánh Giá Đối Đầu Checkpoint P0 (Commit `977b282`)

Notebook này phục vụ quy trình đánh giá đối đầu trực tiếp (**Matched Head-to-Head Evaluation**) giữa:
- **`candidate_p0`**: Checkpoint tốt nhất của ứng viên P0 (`pilot_20261109_182359/p0_legacy_full/best.pth`, hash `9d97613c…`).
- **`incumbent_refit`**: Trọng số mô hình hiện hành (`fatformer_anchored_v2_best.pth`, hash `5cd49252…`), được hiệu chuẩn lại (*refit calibration*) độc lập theo cùng phương pháp để đảm bảo so sánh công bằng.

### Yêu cầu trước khi chạy:
1. **Môi trường Colab**: Chọn GPU CUDA (Runtime -> Change runtime type -> T4, L4, hoặc A100).
2. **Google Drive**: Thư mục `MyDrive/Fatformer/` phải có sẵn **5 tệp** bắt buộc:
   - `phase7/improvement_round1/pilot_20261109_182359/p0_legacy_full/best.pth`
   - `checkpoint/phase7/anchored_v2_run1/fatformer_anchored_v2_best.pth`
   - `datasets/test_communityforensics.tar`
   - `datasets/test_benchmark_gans.tar`
   - `datasets/test_benchmark_diffusion.tar`

### Quy trình chạy:
- Chạy tuần tự **Cell 1** (Mount Drive & GPU) -> **Cell 2** (Git Checkout) -> **Cell 3** (Chạy đánh giá Full Benchmark).
- Sau khi hoàn tất, tệp `evaluation_results_without_weights.zip` sẽ được tải về máy. Hãy giải nén vào `wait/phase7_pilot_evaluation/<EVAL_RUN_ID>/`.

In [1]:
# ==============================================================================
# CELL 1: KẾT NỐI GOOGLE DRIVE & KIỂM TRA GPU RUNTIME
# ==============================================================================
from google.colab import drive
import torch

print("[1/2] Đang kết nối Google Drive (vui lòng cấp quyền trong popup nếu được yêu cầu)...")
drive.mount('/content/drive')

print("\n[2/2] Kiểm tra GPU CUDA:")
assert torch.cuda.is_available(), "LỖI: Chưa chọn GPU! Vui lòng vào Runtime -> Change runtime type -> Chọn GPU (T4, L4 hoặc A100) trước khi chạy."
print(f"✓ Đã phát hiện GPU: {torch.cuda.get_device_name(0)}")
!nvidia-smi

[1/2] Đang kết nối Google Drive (vui lòng cấp quyền trong popup nếu được yêu cầu)...
Mounted at /content/drive

[2/2] Kiểm tra GPU CUDA:
✓ Đã phát hiện GPU: NVIDIA L4
Tue Nov 10 06:58:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   38C    P8             14W /   72W |       3MiB /  23034Mi

In [2]:
# ==============================================================================
# CELL 2: CLONE MÃ NGUỒN TỪ GIT (NHÁNH hoang) & KHÓA COMMIT 977b282
# ==============================================================================
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/fatformer-xla')
PINNED_COMMIT = '977b282'

if not REPO_DIR.exists():
    print("Đang clone repo từ GitHub (nhánh hoang)...")
    !git clone --branch hoang --single-branch https://github.com/LeeVHoangtk3/fatformer-xla.git /content/fatformer-xla
else:
    print(f"Repo đã tồn tại tại {REPO_DIR}. Đang fetch cập nhật...")
    !git -C /content/fatformer-xla fetch origin hoang

print(f"\nKhóa phiên bản tại commit: {PINNED_COMMIT}")
!git -C /content/fatformer-xla checkout {PINNED_COMMIT}

current_commit = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True
).strip()
print(f"\n✓ Commit hiện tại đang chạy: {current_commit}")
assert current_commit.startswith(PINNED_COMMIT), f"Lệch commit! Yêu cầu {PINNED_COMMIT}, thực tế {current_commit}"

Đang clone repo từ GitHub (nhánh hoang)...
Cloning into '/content/fatformer-xla'...
remote: Enumerating objects: 3436, done.
remote: Counting objects: 100% (188/188), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 3436 (delta 80), reused 162 (delta 72), pack-reused 3248 (from 2)
Receiving objects: 100% (3436/3436), 75.33 MiB | 16.58 MiB/s, done.
Resolving deltas: 100% (1937/1937), done.

Khóa phiên bản tại commit: 977b282
Note: switching to '977b282'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEA

In [3]:
# ==============================================================================
# CELL 3: CHẠY TOÀN BỘ ĐÁNH GIÁ ĐỐI ĐẦU (FULL MATCHED EVALUATION)
# ==============================================================================
# Tiến trình này sẽ tự động:
# 1. Xác thực hash SHA-256 của 5 tệp trên Drive (2 checkpoint + 3 archive dữ liệu).
# 2. Giải nén dữ liệu sang SSD Colab (/content/phase7_p0_evaluation/<EVAL_RUN_ID>/data).
# 3. Trích xuất logits và fit calibration riêng cho từng model trên cùng 400 ảnh val_calib.
# 4. Đóng gói checkpoint, kiểm chứng 1.200 tensor và smoke test 20 ảnh thực tế.
# 5. Chạy benchmark 1.600 ảnh CFE (7 điều kiện) và 82.003 ảnh in-dist (18 tập) cho cả 2 model.
# 6. Đo độ trễ suy luận (latency overhead), tính toán Gate G7.2 và paired comparison.
# 7. Tự động sao lưu kết quả về Drive và tải tệp evaluation_results_without_weights.zip về máy.
# ==============================================================================
from pathlib import Path
import subprocess

REPO = Path('/content/fatformer-xla')
EVAL_REPO = str(REPO)
EVAL_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True
).strip()

EVAL_STAGE = 'full'
EVAL_MAX_BATCH = 16

print(f"=== BẮT ĐẦU CHẠY GIAI ĐOẠN ĐÁNH GIÁ: {EVAL_STAGE.upper()} ===")
print(f"Commit: {EVAL_COMMIT}")
print(f"Max Batch Size: {EVAL_MAX_BATCH}")

exec(compile(
    (REPO / 'notebooks/task/phase7_pilot_evaluation/start_colab.py').read_text(),
    'start_colab.py', 'exec'
))

print("\n" + "="*80)
print("✓ ĐÁNH GIÁ ĐỐI ĐẦU ĐÃ HOÀN TẤT THÀNH CÔNG!")
print("Tệp evaluation_results_without_weights.zip đã được lưu trên Drive và bắt đầu tải về.")
print("Sau khi tải về, hãy giải nén vào thư mục: wait/phase7_pilot_evaluation/<EVAL_RUN_ID>/")
print("="*80)

=== BẮT ĐẦU CHẠY GIAI ĐOẠN ĐÁNH GIÁ: FULL ===
Commit: 977b282852120a7c5d1cb5d3cc110955edf00ce7
Max Batch Size: 16
Git commit: 977b282852120a7c5d1cb5d3cc110955edf00ce7 Eval ID: p0_eval_20261110_065901 GPU: NVIDIA L4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Both models are recalibrated separately; incumbent_refit means old weights, not an exact replay of package 24869052.
Hashing dataset archive: /content/drive/MyDrive/Fatformer/datasets/test_communityforensics.tar
Hashing dataset archive: /content/drive/MyDrive/Fatformer/datasets/test_benchmark_gans.tar
Hashing dataset archive: /content/drive/MyDrive/Fatformer/datasets/test_benchmark_diffusion.tar
Staging dataset: test_communityforensics.tar
Staging dataset: test_benchmark_gans.tar
Staging dataset: test_benchmark_diffusion.tar
Results/weights on Drive: /content/drive/MyDrive/Fatformer/phase7/improvement_round1/evaluation/p0_eval_20261110_065901


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Finished. Review G7.2 and matched comparison; no automatic deployment or best-reference G7.3 claim.

✓ ĐÁNH GIÁ ĐỐI ĐẦU ĐÃ HOÀN TẤT THÀNH CÔNG!
Tệp evaluation_results_without_weights.zip đã được lưu trên Drive và bắt đầu tải về.
Sau khi tải về, hãy giải nén vào thư mục: wait/phase7_pilot_evaluation/<EVAL_RUN_ID>/
